# OMS API Query
The goal of this notebook is connecting to the CMS online monitoring system and collecting the numbers/data of all the runs that took place in 2026, are proton-proton collisions and have a stable beam

## Imports

In [41]:
import json
import csv
from omsapi import OMSAPI

## Global Variables & Filters

In [31]:
PER_PAGE = 999 # how many results do we want to see per page
ERA_NAMES = ["Run2026A", "Run2026B", "Run2026C", "Run2026D"]
 
RUN_FILTERS = {
    "fill_type_runtime": "PROTONS",
    "l1_hlt_mode": "collisions2026",
    "stable_beam": True,
}

## Helper function definitions

In [32]:
def connect(verbose=True):
    omsapi = OMSAPI("https://cmsoms.cern.ch/agg/api", "v1",
                    cert_verify=False, verbose=verbose, throw_on_err=True)
    omsapi.auth_krb()
    return omsapi

In [33]:
def _fetch_all(omsapi, resource, filters=None, per_page=PER_PAGE):
    """Walk every page of a resource and return the list of JSON:API objects.
 
    The client has no page walker, so we bump page[offset] until a page comes
    back shorter than the requested limit.
    """
    query = omsapi.query(resource)
    for attribute, value in (filters or {}).items():
        query.filter(attribute, value)
 
    rows = []
    page = 1
    while True:
        query.paginate(page=page, per_page=per_page)
        batch = query.data().json()["data"]
 
        rows.extend(batch)
        if len(batch) < per_page:
            return rows
        page += 1

In [34]:
def get_eras(omsapi, era_names=ERA_NAMES):
    """Fetch all eras, keep the requested ones (drops HI and non-2026 eras)."""
    eras = _fetch_all(omsapi, "eras")
    by_name = {era["attributes"]["name"]: era["attributes"] for era in eras}
    return [by_name[name] for name in era_names]

In [35]:
def get_runs(omsapi, era_name, run_filters=RUN_FILTERS):
    filters = dict(run_filters, era=era_name)
    return [run["attributes"] for run in _fetch_all(omsapi, "runs", filters)]

In [36]:
def get_fill_schemes(omsapi, era_name):
    """Map fill_number -> injection_scheme (the filling scheme lives on fills)."""
    fills = _fetch_all(omsapi, "fills", {"era": era_name})
    return {fill["attributes"]["fill_number"]: fill["attributes"]["injection_scheme"]
            for fill in fills}

In [37]:
def collect(omsapi):
    """Return [{"era": <era attributes>, "runs": [<run attributes>, ...]}]."""
    eras = get_eras(omsapi)
    print("eras: {}".format(", ".join(era["name"] for era in eras)))
 
    collected = []
    for era in eras:
        runs = get_runs(omsapi, era["name"])
        fills = sorted({run["fill_number"] for run in runs})
        schemes = get_fill_schemes(omsapi, era["name"]) 
        for run in runs:
            run["injection_scheme"] = schemes.get(run["fill_number"]) # query injection scheme based on fill number
        print("=== era {}contains {} runs in {} fills===".format(
            era["name"], len(runs), len(fills)))
        print("  fills: {}".format(fills))
        print("  runs:  {}".format(sorted(run["run_number"] for run in runs)))
 
        collected.append({"era": era, "runs": runs})
 
    return collected

In [38]:
def save_everything(eras, path="oms_runs.json"):
    with open(path, "w") as handle:
        json.dump(eras, handle, indent=2) # saves the full json response for each run

In [43]:
def save_csv(eras, path="oms_runs.csv"):
    with open(path, "w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["run_number", "injection_scheme", "Run era"])
        for era in eras:
            for run in era["runs"]:
                writer.writerow([run["run_number"], run["injection_scheme"], run["era"]])
 

## Executing the pipeline

In [44]:
save_csv(collect(connect()))

https://cmsoms.cern.ch/agg/api/v1/eras?page[offset]=0&page[limit]=999
eras: Run2026A, Run2026B, Run2026C, Run2026D
https://cmsoms.cern.ch/agg/api/v1/runs?filter[fill_type_runtime][EQ]=PROTONS&filter[l1_hlt_mode][EQ]=collisions2026&filter[stable_beam][EQ]=True&filter[era][EQ]=Run2026A&page[offset]=0&page[limit]=999
https://cmsoms.cern.ch/agg/api/v1/fills?filter[era][EQ]=Run2026A&page[offset]=0&page[limit]=999
=== era Run2026Acontains 37 runs in 9 fills===
  fills: [11471, 11472, 11474, 11475, 11477, 11479, 11481, 11496, 11498]
  runs:  [401623, 401624, 401625, 401626, 401627, 401630, 401631, 401632, 401633, 401634, 401635, 401636, 401637, 401638, 401639, 401640, 401641, 401642, 401661, 401663, 401664, 401668, 401669, 401670, 401671, 401691, 401692, 401693, 401694, 401699, 401730, 401731, 401732, 401733, 401803, 401831, 401832]
https://cmsoms.cern.ch/agg/api/v1/runs?filter[fill_type_runtime][EQ]=PROTONS&filter[l1_hlt_mode][EQ]=collisions2026&filter[stable_beam][EQ]=True&filter[era][EQ]=R